In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pylab as plt
import networkx as nx
from sklearn.metrics import pairwise_distances
from sklearn.decomposition import PCA

In [ ]:
from modules import geometry
from modules import graphio
from modules import metrics
from modules import visualization as vis
from modules.projector import MDSTorusProjector

In [ ]:
# apsp_distance_matrix / get_periodic_lattice  →  graphio module

## Lattice graph

In [ ]:
G, D = graphio.get_periodic_lattice(20, 20)

# D /= np.max(D)

proj = MDSTorusProjector()
proj.fit_transform(D)

fig, ax = plt.subplots()
vis.plot_embedding_with_torus_edges(G=G, torus=proj, ax=ax)

### 20×20 grid: comparing learn modes

In [ ]:
G20, D20 = graphio.get_periodic_lattice(20, 20)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, mode in zip(axes, ['alpha', 'square', 'rectangular']):
    proj = MDSTorusProjector()
    proj.fit_transform(D20, learn_mode=mode)
    if mode == 'alpha':
        print(f"[{mode}]  alpha={proj.alpha_:.4g}")
    elif mode == 'square':
        print(f"[{mode}]  r={proj.r0_:.4g}")
    else:
        print(f"[{mode}]  r0={proj.r0_:.4g}  r1={proj.r1_:.4g}  ratio={proj.r1_/proj.r0_:.3f}")
    vis.plot_embedding_with_torus_edges(G=G20, torus=proj, ax=ax)
    ax.set_title(f"learn_mode='{mode}'")
plt.suptitle("20×20 periodic grid")
plt.tight_layout()

## 20×40 periodic grid

In [ ]:
G40, D40 = graphio.get_periodic_lattice(20, 40)
colors = [j for (i, j) in G40.nodes()]
colors_short = [i for (i, j) in G40.nodes()]

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, mode in zip(axes, ['alpha', 'square', 'rectangular']):
    proj = MDSTorusProjector()
    proj.fit_transform(D40, learn_mode=mode)
    if mode == 'alpha':
        print(f"[{mode}]  alpha={proj.alpha_:.4g}")
    elif mode == 'square':
        print(f"[{mode}]  r={proj.r0_:.4g}")
    else:
        print(f"[{mode}]  r0={proj.r0_:.4g}  r1={proj.r1_:.4g}  ratio={proj.r1_/proj.r0_:.3f}")
    vis.plot_embedding_with_torus_edges(G=G40, torus=proj, ax=ax, colors=colors)
    ax.set_title(f"learn_mode='{mode}'")
plt.suptitle("20×40 periodic grid")
plt.tight_layout()

### 20×40 grid: fixed 1:2 ratio, learn scale only (sanity check)

In [ ]:
colors = [j for (i, j) in G40.nodes()]

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

proj = MDSTorusProjector()
proj.fit_transform(D40, learn_mode='alpha', r0_init=1.0, r1_init=2.0)
print(f"[fixed ratio]    alpha={proj.alpha_:.4g}  r0={proj.r0_:.4g}  r1={proj.r1_:.4g}")
vis.plot_embedding_with_torus_edges(G=G40, torus=proj, ax=axes[0], colors=colors)
axes[0].set_title("fixed ratio 1:2, learned alpha")

proj = MDSTorusProjector()
proj.fit_transform(D40, learn_mode='rectangular', r0_init=25.0, r1_init=50.0)
print(f"[learned ratio]  alpha={proj.alpha_:.4g}  r0={proj.r0_:.4g}  r1={proj.r1_:.4g}  ratio={proj.r1_/proj.r0_:.3f}")
vis.plot_embedding_with_torus_edges(G=G40, torus=proj, ax=axes[1], colors=colors)
axes[1].set_title(f"learned ratio (init 1:3)  r0={proj.r0_:.2g}  r1={proj.r1_:.2g}")

plt.tight_layout()

## Grid cells

In [ ]:
a = np.load("grid_cells_4800.npz")
data = a['data']
spikes = a['spikes']

D = pairwise_distances(data)
X = MDSTorusProjector().fit_transform(D)

In [ ]:
# spatial location of firing rates
fig, ax = plt.subplots(1, 5, figsize=(10, 3))

cell_ids = [5, 39, 75, 109, 115] # cells with clearly structured firing pattern

xx = X[:,0]
yy = X[:,1]
for i, idx in enumerate(cell_ids):
    order  = np.argsort(spikes[:, idx])  # to plot interesting time points on top
    ax[i].scatter(xx[order], yy[order], c=spikes[order, idx], s=2)
    ax[i].set_title(f"Cell {idx}")
    ax[i].axis("off")
    ax[i].set_aspect("equal")
fig.suptitle("Firing rates of selected cells by spatial position\n Toroidal embedding")

## Yoda and Bulldog

In [ ]:
pca = np.load("yoda_bulldog_pca50.npy")

D = pairwise_distances(pca[:,:10])
X = MDSTorusProjector().fit_transform(D)

In [ ]:
fig, ax = plt.subplots()
plt.scatter(X[:,0], X[:,1],c=pca[:,1],s=5)

## Block models

In [ ]:
G = nx.planted_partition_graph(5, 100, 0.2, 0.01)
d, nodes = graphio.apsp_distance_matrix(G)

X_torus = MDSTorusProjector().fit_transform(d)

In [ ]:
colors = [0] * 100 + [1] * 100 + [2] * 100 + [3] * 100 + [4] * 100
vis.plot_embedding_with_torus_edges(X_torus, G, colors=colors)

## Rhombic torus (theta = 60°)

In [ ]:
G20, D20 = graphio.get_periodic_lattice(20, 20)
colors20 = [i for (i, j) in G20.nodes()]

proj = MDSTorusProjector()
proj.fit_transform(D20, learn_mode='alpha', theta=60.0)
print(f"alpha={proj.alpha_:.4g}  r0={proj.r0_:.4g}  r1={proj.r1_:.4g}")

fig, ax = plt.subplots()
vis.plot_embedding_with_torus_edges(G=G20, torus=proj, ax=ax, colors=colors20)
ax.set_title("20×20 grid, rhombic torus θ=60°")

In [ ]:
K7 = nx.complete_graph(7)
D_K7, _ = graphio.apsp_distance_matrix(K7)

proj_k7 = MDSTorusProjector()
proj_k7.fit_transform(D_K7, learn_mode='alpha', theta=60.0)
print(f"alpha={proj_k7.alpha_:.4g}  r0={proj_k7.r0_:.4g}  r1={proj_k7.r1_:.4g}")

fig, ax = plt.subplots()
vis.plot_embedding_with_torus_edges(G=K7, torus=proj_k7, ax=ax, colors=list(range(7)))
ax.set_title("K7, rhombic torus θ=60°")

In [ ]:
k = 50
G_sbm = nx.planted_partition_graph(7, k, 0.5, 0.01, seed=0)
D_sbm, _ = graphio.apsp_distance_matrix(G_sbm)

proj_sbm = MDSTorusProjector()
proj_sbm.fit_transform(D_sbm, learn_mode='alpha', theta=60.0)
print(f"alpha={proj_sbm.alpha_:.4g}  r0={proj_sbm.r0_:.4g}  r1={proj_sbm.r1_:.4g}")

colors_sbm = [b for b in range(7) for _ in range(k)]
fig, ax = plt.subplots()
vis.plot_embedding_with_torus_edges(G=G_sbm, torus=proj_sbm, ax=ax, colors=colors_sbm)
ax.set_title("7-block SBM (7×50 nodes), rhombic torus θ=60°")